# Siegle, Jia et al. (2021): the cross-correlogram claims

**Paper.** Siegle JH, Jia X, et al. (2021). Survey of spiking in the mouse visual system reveals
functional hierarchy. *Nature* 592, 86-92.
[doi:10.1038/s41586-020-03171-x](https://doi.org/10.1038/s41586-020-03171-x). Claims file written
from the PMC author manuscript (PMC10399640).

**Data.** The target dandiset, [DANDI:000253](https://dandiarchive.org/dandiset/000253/0.240923.1441)
(version 0.240923.1441), holds a different experiment (see `reproduce_siegle_2021.ipynb`). The CCG
analyses use the Functional Connectivity sessions, which the paper deposited as
[DANDI:000022](https://dandiarchive.org/dandiset/000022/0.251116.2247), version 0.251116.2247, its
only published version. The paper's per-unit layer table (`data/layer_info.csv` of
AllenInstitute/neuropixels_platform_paper at commit 4f8bcc26) labels layers for ed10ce; its link to
the dandisets' unit ids is tested in the other notebook (link-unit-tables).

| Claim | Here |
|---|---|
| fig2-sharp-peak-pairs | tested |
| fig2d-v1-leads-lm-example | tested |
| fig2-neighbour-offsets-across-mice | tested |
| ed7c-v1-leads-am-follows | tested |
| fig2f-directionality-vs-anatomy | tested |
| ed7e-distance-does-not-explain | blocked: the left-right CCF coordinate is not in the files |
| ed7fg-absent-in-spontaneous | tested |
| ed8-total-hierarchy-score | tested |
| ed10ce-superficial-lead-deep | tested |

Method (siegle.py, "cross-correlograms"): 1 ms bins; per orientation (0, 45, 90, 135 degrees,
contrast 0.8, 75 trials) the CCG of every pair of units of V1, LM, RL, AL, PM and AM with a mean
rate above 2 Hz from 50 to 500 ms after onset, normalized by the triangular overlap and the
geometric mean rate, minus its expected value under jitter of both trains within 25 ms windows,
averaged over the four orientations; a sharp peak when the maximum within +/-10 ms exceeds 7 times
the SD of the flanks at 50 to 100 ms. Readings named by the claims file: window 2 s (the
presentation) or 3 s (presentation and grey); units (a) all non-noise, (b) with the Extended Data
Fig. 1j filters (ISI violations < 0.5, amplitude cutoff < 0.1, presence ratio > 0.95), which only
removes pairs. The spontaneous period (30 min grey screen) is cut into consecutive 2 s or 3 s
segments, with the same units. Four sessions run at a time, about 80 s each.

In [ ]:
import os
os.environ.setdefault("OMP_NUM_THREADS", "1")
os.environ.setdefault("OPENBLAS_NUM_THREADS", "1")
import io
import time
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import requests
from scipy import stats

import siegle as S

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 40)
warnings.filterwarnings("ignore", category=RuntimeWarning)
started = time.time()
assets = S.list_assets("000022", S.SOURCES["000022"])
fc = assets[~assets.path.str.contains("probe")].reset_index(drop=True)
res = S.load_ccg_all(fc)
layer_csv = pd.read_csv(io.StringIO(requests.get(f"{S.CODE_RAW}/data/layer_info.csv", timeout=60).text))
print(f"{len(res)} sessions in {(time.time() - started) / 60:.1f} min")

## Data check

In [ ]:
info = pd.DataFrame([{"subject": r["subject"], "session": r["session"], "units_above_2Hz": len(r["units"]),
                      "dg_trials_contrast_0.8": r["n_dg_trials"], "orientations": len(r["orientations"]),
                      "dg_onset_interval_s": round(r["dg_onset_interval"], 3),
                      "spont_s": round(r.get("spont_duration", np.nan), 1),
                      "segments_2s": r.get("n_spont_segments_2000"), "seconds": round(r["seconds"])} for r in res])
print(info.to_string(index=False))
print("\nmice:", info.subject.nunique(), " units:", info.units_above_2Hz.sum())
allu = pd.concat([r["units"].assign(subject=r["subject"]) for r in res])
print("units above 2 Hz by area:", allu.location.value_counts().reindex(S.CORTEX).to_dict())
print("rate from 50 to 500 ms (Hz), range:", round(allu.rate_dg.min(), 2), "to", round(allu.rate_dg.max(), 2))

Every session has 300 trials at contrast 0.8 over the four orientations (75 each); contrast 0.8
and 0.1 trials alternate, so successive 0.8 onsets are 6 s apart and each trial has 2 s of
grating and at least 1 s of grey after it. The 30 min spontaneous block is present in each session.
The paper reports 25 mice for the CCG analyses; the dandiset holds 26 Functional Connectivity
sessions and nothing identifies the one left out, so all 26 are used.

In [ ]:
AREA_ORDER = ["VISp", "VISl", "VISrl", "VISal", "VISpm", "VISam"]  # anatomical order of the cortex
NAMES = {a: S.NAMES[a] for a in AREA_ORDER}
HSC = np.array([S.HIERARCHY[a] for a in AREA_ORDER])


def ed1j(u):
    return ((u.isi_violations < 0.5) & (u.amplitude_cutoff < 0.1) & (u.presence_ratio > 0.95)).values


def peak_table(reading, quality):
    # one row per significant pair and direction: source unit, target unit, offset (source leads when > 0)
    rows, npairs = [], []
    for r in res:
        u = r["units"]
        ok = np.ones(len(u), bool) if quality == "all" else ed1j(u)
        p = r["peaks"][reading]
        p = p[ok[p.i.values] & ok[p.j.values]]
        n_ok = ok.sum()
        npairs.append({"subject": r["subject"], "pairs": n_ok * (n_ok - 1) // 2, "peaks": len(p)})
        loc = u.location.values
        for sgn, a, b in ((1, "i", "j"), (-1, "j", "i")):
            rows.append(pd.DataFrame({"subject": r["subject"], "src": loc[p[a].values], "tgt": loc[p[b].values],
                                      "src_id": u.unit_id.values[p[a].values], "tgt_id": u.unit_id.values[p[b].values],
                                      "offset": sgn * p.offset.values}))
    return pd.concat(rows, ignore_index=True), pd.DataFrame(npairs)


def ds(offsets, zero="exclude"):
    pos = np.sum(offsets > 0) + (np.sum(offsets == 0) if zero == "positive" else 0)
    neg = np.sum(offsets < 0)
    return (pos - neg) / (pos + neg) if pos + neg else np.nan


def ds_matrix(t, mode="pooled", zero="exclude"):
    # rows: target, columns: source (as Figure2/ccg_distribution_model_final.ipynb)
    M = np.full((6, 6), np.nan)
    for i, tg in enumerate(AREA_ORDER):
        for j, sr in enumerate(AREA_ORDER):
            d = t[(t.src == sr) & (t.tgt == tg)]
            if mode == "pooled":
                M[i, j] = ds(d.offset.values, zero)
            else:
                M[i, j] = np.nanmean([ds(g.offset.values, zero) for _, g in d.groupby("subject")]) if len(d) else np.nan
    return M


def anatomy_corr(M):
    A = HSC[:, None] - HSC[None, :]  # target score minus source score
    il = np.tril_indices(6)
    x, y = A[il], M[il]
    ok = np.isfinite(y)
    r, p = stats.pearsonr(x[ok], y[ok])
    return r, p, int(ok.sum())


READINGS = [(w, q) for w in ("grating_2000", "grating_3000") for q in ("all", "ed1j")]
tables = {rq: peak_table(*rq) for rq in READINGS}

## fig2-sharp-peak-pairs

> Jitter-corrected CCGs during drifting gratings showed a significant sharp peak for 16,119 of
> 2,089,890 possible cortical unit pairs, 0.96% +/- 0.13% per mouse in 25 mice.

In [ ]:
rows = []
for (w, q), (t, n) in tables.items():
    frac = 100 * n.peaks / n.pairs
    rows.append({"window": w, "units": q, "peaks": int(n.peaks.sum()), "pairs": int(n.pairs.sum()),
                 "per_mouse_mean_%": frac.mean(), "per_mouse_sd_%": frac.std(ddof=0),
                 "per_mouse_sem_%": frac.std() / np.sqrt(len(frac)), "mice": len(n),
                 "pooled_%": 100 * n.peaks.sum() / n.pairs.sum()})
summ = pd.DataFrame(rows)
print(summ.round(4).to_string(index=False))
print("expected: 16119 +/- 1612 peaks, 2089890 +/- 208989 pairs, 0.96 +/- 0.13 % per mouse")
n0 = tables[("grating_2000", "all")][1]
print("\nper mouse, 2 s, all non-noise units:")
print(n0.assign(percent=(100 * n0.peaks / n0.pairs).round(3)).to_string(index=False))
t0 = tables[("grating_2000", "all")][0]
print("\noffset distribution of all sharp peaks (ms, one direction per pair):",
      t0[t0.offset >= 0].offset.describe().round(2).to_dict())

**Result: reproduced.** With 2 s windows and all non-noise units (readings a, a), 17,592 sharp
peaks of 2,116,259 possible pairs (paper 16,119 of 2,089,890), and 0.94% per mouse (paper 0.96%),
over 26 mice: all three within tolerance. The SD across mice is 0.57%; the s.e.m. printed above is
near the paper's 0.13%. With the Extended Data Fig. 1j filters the possible pairs fall to 544,044,
far from the paper's count, so the paper's CCG units were not filtered that way; the 3 s window
gives 18,226 peaks, outside tolerance.

## fig2d-v1-leads-lm-example

> In an example mouse, the CCG peak offsets between V1 and LM were shifted positive against the V1-V1
> distribution (median 3.9 ms), so V1 units on average lead LM units.

In [ ]:
def per_mouse_test(t, src, tgt, ref_src, ref_tgt):
    out = []
    for subj, g in t.groupby("subject"):
        a = g[(g.src == src) & (g.tgt == tgt)].offset.values
        b = g[(g.src == ref_src) & (g.tgt == ref_tgt)].offset.values
        if len(a) and len(b):
            out.append({"subject": subj, "n": len(a), "n_ref": len(b), "median": np.median(a),
                        "p": stats.ranksums(a, b)[1], "higher": np.median(a) > np.median(b)})
    return pd.DataFrame(out)


for rq in READINGS:
    d = per_mouse_test(tables[rq][0], "VISp", "VISl", "VISp", "VISp")
    hit = d[(d["median"].sub(3.9).abs() <= 0.39) & (d.p < 0.05) & d.higher]
    print(f"{rq}: mice with both distributions {len(d)}; matching mice (median 3.51 to 4.29 ms, p < 0.05, higher): {len(hit)}")
    if rq == READINGS[0]:
        print(d.round(5).to_string(index=False))
    if len(hit):
        print(hit.round(6).to_string(index=False))

**Result: reproduced.** In 4 of the 15 mice with both distributions (2 s, all units) the median
V1 to LM offset is 4 ms and the V1-LM offsets lie above the V1-V1 offsets (rank-sum p < 0.002);
14 of the 15 medians are positive. Every reading has at least two such mice.

## fig2-neighbour-offsets-across-mice

> Across 25 mice, the CCG peak offset distributions between neighbouring cortical areas differ from
> the within-area distributions for V1-LM, LM-RL, RL-AL and PM-AM, but not for AL-PM.

Per mouse two-sided rank-sum of the source-to-target offsets (lower area as source) against the
lower area's within-area offsets, combined over mice with Fisher's method.

In [ ]:
NEIGH = [("VISp", "VISl"), ("VISl", "VISrl"), ("VISrl", "VISal"), ("VISal", "VISpm"), ("VISpm", "VISam")]
expect = {("VISal", "VISpm"): False}
for rq in READINGS:
    rows = []
    for a, b in NEIGH:
        d = per_mouse_test(tables[rq][0], a, b, a, a)
        stat, p = stats.combine_pvalues(d.p.values, method="fisher") if len(d) else (np.nan, np.nan)
        rows.append({"pair": f"{NAMES[a]}-{NAMES[b]}", "mice": len(d), "fisher_p": p, "significant": p < 0.05,
                     "expected": expect.get((a, b), True), "median_of_mouse_medians": d["median"].median() if len(d) else np.nan})
    t = pd.DataFrame(rows)
    print(rq, " agreeing pairs:", int((t.significant == t.expected).sum()), "of 5")
    print(t.to_string(index=False))

**Result: partially reproduced.** With 2 s windows and all non-noise units, V1-LM, LM-RL, RL-AL
and PM-AM differ from the within-area distributions (Fisher p from 2e-103 to 1.4e-05), as stated,
but AL-PM also differs (p = 1.8e-05, 8 mice): 4 of 5 pairs agree. With the Extended Data Fig. 1j
filters and 2 s windows AL-PM is not significant (p = 0.057, 4 mice) and all 5 agree; the 3 s
readings give 4 of 5.

## ed7c-v1-leads-am-follows

> On average V1 units lead the units of the other cortical areas, while AM follows the other areas.

Per mouse and ordered pair, the median offset; averaged across mice.

In [ ]:
def mean_median_matrix(t):
    M = pd.DataFrame(np.nan, index=[NAMES[a] for a in AREA_ORDER], columns=[NAMES[a] for a in AREA_ORDER])
    for sr in AREA_ORDER:
        for tg in AREA_ORDER:
            d = t[(t.src == sr) & (t.tgt == tg)]
            if len(d):
                M.loc[NAMES[sr], NAMES[tg]] = d.groupby("subject").offset.median().mean()
    return M  # rows: source, columns: target


for rq in READINGS:
    M = mean_median_matrix(tables[rq][0])
    v1 = M.loc["V1", ["LM", "RL", "AL", "PM", "AM"]]
    am = M.loc[["V1", "LM", "RL", "AL", "PM"], "AM"]
    print(f"{rq}: V1 as source positive for {int((v1 > 0).sum())} of 5; AM as target positive for {int((am > 0).sum())} of 5")
    if rq == READINGS[0]:
        print("mean over mice of the median offset (ms), rows source, columns target:")
        print(M.round(3).to_string())

**Result: partially reproduced.** With 2 s windows and all units, the mean median offset is
positive from V1 to each of the five other areas (0.28 to 4.58 ms) and into AM from four of the
five (1.63 to 4.58 ms); from PM to AM it is -1.54 ms. 9 of 10 pairs hold; the 3 s window with the
Extended Data Fig. 1j filters gives 10 of 10.

## fig2f-directionality-vs-anatomy

> The matrix of pairwise directionality scores between cortical areas is correlated with the matrix
> of anatomical hierarchy score differences (Pearson r = 0.74 over 21 pairs).

DS from offsets pooled over mice (zero lags in neither count), rows target and columns source;
anatomy target minus source; the 15 lower-triangle and 6 diagonal entries. Variants: DS averaged
across mice; zero lags counted positive; the paper's precomputed matrix `FFscore_grating_10.npy`.

In [ ]:
fig2f = {}
for rq in READINGS:
    t = tables[rq][0]
    for mode in ("pooled", "mice"):
        for zero in ("exclude", "positive"):
            M = ds_matrix(t, mode, zero)
            r, p, n = anatomy_corr(M)
            fig2f[(rq, mode, zero)] = (r, p, n, M)
print(pd.DataFrame([{"window": k[0][0], "units": k[0][1], "DS": k[1], "zero lag": k[2], "r": v[0], "p": v[1], "n": v[2]}
                    for k, v in fig2f.items()]).round(5).to_string(index=False))
print("expected r = 0.74 +/- 0.074, p < 0.05")
M0 = fig2f[(READINGS[0], "pooled", "exclude")][3]
print("\nDS matrix, 2 s, all units, pooled (rows target, columns source):")
print(pd.DataFrame(M0, index=[NAMES[a] for a in AREA_ORDER], columns=[NAMES[a] for a in AREA_ORDER]).round(3).to_string())
ff = np.load(io.BytesIO(requests.get(f"{S.CODE_RAW}/data/processed_data/FFscore_grating_10.npy", timeout=60).content))[0]
code_order = ["VISp", "VISrl", "VISl", "VISal", "VISpm", "VISam"]
hs_code = np.array([S.HIERARCHY[a] for a in code_order])
Acode = hs_code[:, None] - hs_code[None, :]
il = np.tril_indices(6)
print("\nvariant, the paper's precomputed DS matrix (its own area order, V1 RL LM AL PM AM): r = %.4f, p = %.3g"
      % stats.pearsonr(Acode[il], ff[il]))

**Result: reproduced.** With 2 s windows, all units and offsets pooled over mice, r = 0.809,
p = 1e-05 over the 21 entries (paper 0.74; tolerance 0.666 to 0.814). The other readings, with DS pooled
or averaged across mice and zero lags excluded, give r = 0.681 to 0.825 (p < 0.001); the 3 s window
with all units and pooled DS (0.825) is just above the tolerance. The paper's precomputed DS matrix gives r = 0.735 with the same anatomy differences,
which confirms their orientation.

## ed7fg-absent-in-spontaneous

> The functional hierarchy is absent during spontaneous activity (30 min grey screen).

Sharp peaks over 2 s or 3 s segments of the spontaneous block, the same units; DS averaged across
mice; Pearson against the hierarchy differences over the 21 entries, at p < 0.05.

In [ ]:
for w in ("spont_2000", "spont_3000"):
    for q in ("all", "ed1j"):
        t, n = peak_table(w, q)
        M = ds_matrix(t, "mice")
        r, p, k = anatomy_corr(M)
        Mp = ds_matrix(t, "pooled")
        rp, pp, kp = anatomy_corr(Mp)
        print(f"{w}, {q}: sharp peaks {int(n.peaks.sum())} of {int(n.pairs.sum())} pairs; per mouse {n.peaks.min()} to {n.peaks.max()} "
              f"(median {n.peaks.median():.0f}); DS averaged across mice r = {r:.4f}, p = {p:.3g} (n = {k}); pooled r = {rp:.4f}, p = {pp:.3g}")

**Result: not reproduced.** During the spontaneous block the DS matrix averaged across mice is
correlated with the hierarchy differences: r = 0.540, p = 0.012 with 2 s segments and r = 0.603,
p = 0.004 with 3 s segments (all units), and p < 0.05 in all eight variants printed. Sharp peaks
are fewer than during gratings (6,354 against 17,592 of the same 2,116,259 pairs).

## ed8-total-hierarchy-score

> The measured functional network has a total hierarchy score of 0.89.

Mean DS of each source over all targets including its own diagonal entry (np.mean(FF_score,
axis=0)); THS the largest minus the smallest.

In [ ]:
for rq in READINGS:
    for mode in ("pooled", "mice"):
        M = fig2f[(rq, mode, "exclude")][3]
        col = np.nanmean(M, axis=0)
        ths = np.nanmax(col) - np.nanmin(col)
        Mn = M.copy()
        np.fill_diagonal(Mn, np.nan)
        coln = np.nanmean(Mn, axis=0)
        print(f"{rq}, DS {mode}: THS = {ths:.4f} (without diagonal {np.nanmax(coln) - np.nanmin(coln):.4f}); "
              f"source means {dict(zip([NAMES[a] for a in AREA_ORDER], np.round(col, 3)))}")
print("expected 0.89 +/- 0.089, between 0.15 and 1.54")

**Result: reproduced.** With 2 s windows, all units and pooled offsets, THS = 0.904 (paper 0.89),
between the one-to-all 0.15 and the feedforward 1.54. V1 has the highest mean DS as a source (0.430)
and AM the lowest (-0.474). Averaging DS across mice gives 0.699, and the 3 s window 1.036.

## ed10ce-superficial-lead-deep

> Within an area, superficial layers (2 to 4) are hierarchically lower than deep layers (5 and 6),
> with superficial layers tending to drive deep layers in the CCG analysis.

Layers from the paper's table (codes 2 and 4 superficial, 5 and 6 deep); pairs within one area,
superficial unit as source. (a) mean over the six areas of each area's DS averaged across mice;
(b) DS of the offsets pooled over the six areas, averaged across mice.

In [ ]:
lay = layer_csv.set_index("ecephys_unit_id").cortical_layer
for rq in READINGS:
    t = tables[rq][0].copy()
    t["layer_src"] = t.src_id.map(lay)
    t["layer_tgt"] = t.tgt_id.map(lay)
    sd = t[(t.src == t.tgt) & t.layer_src.isin([2, 4]) & t.layer_tgt.isin([5, 6])]
    per_area = {}
    for a in AREA_ORDER:
        g = sd[sd.src == a]
        per_area[NAMES[a]] = np.nanmean([ds(x.offset.values) for _, x in g.groupby("subject")]) if len(g) else np.nan
    pooled = np.nanmean([ds(x.offset.values) for _, x in sd.groupby("subject")])
    print(f"{rq}: pairs superficial-deep within an area {len(sd)}; (a) mean over areas {np.nanmean(list(per_area.values())):.4f}; "
          f"(b) pooled over areas, mean over mice {pooled:.4f}; per area {dict((k, round(v, 3)) for k, v in per_area.items())}")
print("unit ids without a layer in the table:", int(pd.concat([r["units"] for r in res]).unit_id.map(lay).isna().sum()))

**Result: reproduced.** The superficial-to-deep DS within an area is positive under both readings
and all four unit and window readings: (a) 0.366 and (b) 0.343 with 2 s windows and all units,
from 3,878 pairs, and positive in each of the six areas (0.12 to 0.63).

## ed7e-distance-does-not-explain

Blocked. The claim needs the 3D CCF distance between units, and in every session of 000021 and
000022 the electrodes' `y` and `z` columns are identical, so the left-right coordinate is missing
(shown in `reproduce_siegle_2021.ipynb`). The areas lie side by side across the cortex, so the
distance cannot be computed without it.

In [ ]:
print(f"total run time {(time.time() - started) / 60:.1f} min")

## Summary

| Claim | Outcome |
|---|---|
| fig2-sharp-peak-pairs | reproduced (2 s, all non-noise units) |
| fig2d-v1-leads-lm-example | reproduced |
| fig2-neighbour-offsets-across-mice | partially reproduced (AL-PM also differs); rests on the adopted unit set |
| ed7c-v1-leads-am-follows | partially reproduced (9 of 10); rests on the adopted unit set |
| fig2f-directionality-vs-anatomy | reproduced, r = 0.809; rests on the adopted window |
| ed7e-distance-does-not-explain | blocked: the left-right CCF coordinate is missing |
| ed7fg-absent-in-spontaneous | not reproduced: the spontaneous DS matrix correlates with the hierarchy |
| ed8-total-hierarchy-score | reproduced, THS 0.904; rests on the adopted window |
| ed10ce-superficial-lead-deep | reproduced |

A failure to reproduce here may be a failure of this attempt, not of the paper: see the
reasoning and the differences from the paper's method.

How the analysis differs from the paper's. The quality filter of the paper's CCG units and the
CCG window are not stated; 2 s windows with all non-noise units are adopted because they give the
paper's number of possible pairs and sharp peaks, and every reading is shown. The jitter
correction uses the exact expected value under jitter of both trains within 25 ms windows; the
paper's code subtracts one random jitter resample. The dandiset has 26 Functional Connectivity
mice, against the paper's 25. The spontaneous block is cut into consecutive 2 s or 3 s segments
and uses the units selected during gratings, choices the paper does not describe.